In [1]:
# %pip install langchain_openai langchain_community langchain pymysql chromadb -q

In [1]:
db_user = "root"
db_host = "localhost"
db_password = "lavi9755"
db_name = "classicmodels"

In [5]:
from langchain_community.utilities.sql_database import SQLDatabase
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")


In [6]:
# %pip install google_generativeai

In [7]:

import google.generativeai as genai
import pymysql

from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# Gemini setup
genai.configure(api_key= GOOGLE_API_KEY )
model = genai.GenerativeModel("gemini-2.5-flash")

# DB connection
conn = pymysql.connect(
    host="localhost",
    user="root",
    password= db_password,
    database= db_name
)
cursor = conn.cursor()

In [8]:
print(db.dialect)
print(db.get_usable_table_names)
print(db.table_info)

mysql
<bound method SQLDatabase.get_usable_table_names of <langchain_community.utilities.sql_database.SQLDatabase object at 0x0000025E13CF9AD0>>

CREATE TABLE sales_raw (
	order_id VARCHAR(10), 
	customer_name VARCHAR(100), 
	product_category VARCHAR(50), 
	order_date TEXT, 
	quantity TEXT, 
	price_per_unit TEXT, 
	country VARCHAR(50)
)COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB

/*
3 rows from sales_raw table:
order_id	customer_name	product_category	order_date	quantity	price_per_unit	country
O001	   John Doe   	Electronics	2025-08-10	2	500	India
O002	Jane Smith	HomeAppliance	08/11/2025	3	1200	 United States 
O003	John Doe	Electronics	2025-08-12	two	500	India
*/


In [9]:
# %pip install langchain-google-genai

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [11]:
def get_schema():
    schema = ""
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()

    for (table_name,) in tables:
        cursor.execute(f"DESCRIBE {table_name}")
        columns = cursor.fetchall()

        schema += f"\nTable: {table_name}\n"
        for col in columns:
            schema += f"{col[0]} ({col[1]})\n"

    return schema

In [12]:
print(get_schema())


Table: sales_raw
order_id (varchar(10))
customer_name (varchar(100))
product_category (varchar(50))
order_date (text)
quantity (text)
price_per_unit (text)
country (varchar(50))



In [13]:
def generate_query(question):
    schema = get_schema()

    prompt = f"""
You are an expert MySQL developer.

Database schema:
{schema}

Rules:
- Only generate SELECT queries
- Use correct table and column names
- Do not explain anything

Question: {question}

SQL Query:
"""

    response = model.generate_content(prompt)

    query = response.text.strip()
    query = query.replace("```sql", "").replace("```", "").strip()

    return query

In [14]:
def run_query(query):
    if any(word in query.lower() for word in ["drop", "delete", "update", "insert"]):
        return "🚫 Unsafe query blocked"

    try:
        cursor.execute(query)
        return cursor.fetchall()
    except Exception as e:
        return f"Error: {str(e)}"

In [15]:
def ask(question):
    query = generate_query(question)
    print("Generated SQL:", query)

    result = run_query(query)

    return result

In [16]:
# %pip install dotenv

In [17]:
# import google.generativeai as genai
# from dotenv import load_dotenv
# import os

# load_dotenv()
# GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# genai.configure(api_key=GOOGLE_API_KEY)

# for m in genai.list_models():
#     print(m.name, m.supported_generation_methods)

In [18]:
print(ask("How many records are there?"))

Generated SQL: SELECT COUNT(*) FROM sales_raw;
((7,),)


In [20]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

answer_prompt = PromptTemplate.from_template(
     """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
 Answer: """
 )

rephrase_answer = answer_prompt | llm | StrOutputParser()

chain = (
     RunnablePassthrough.assign(query=generate_query).assign(
         result=itemgetter("query") | RunnableLambda(run_query)
     )
     | rephrase_answer
 )

chain.invoke({"question": "what product category Jane smith is buying"})


'Jane Smith is buying products from the HomeAppliance category.'

### creating prompt and chain

In [23]:
chain.get_prompts()[0].pretty_print()

Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
 Answer: 


In [24]:
examples = [

    {
        "input": "List all unique customer names after removing extra spaces.",
        "query": "SELECT DISTINCT TRIM(customer_name) FROM orders;"
    },

    {
        "input": "Find all orders where the country is India.",
        "query": "SELECT * FROM orders WHERE TRIM(country) = 'India';"
    },

    {
        "input": "Get all valid orders with proper date format YYYY-MM-DD.",
        "query": "SELECT * FROM orders WHERE order_date REGEXP '^[0-9]{4}-[0-9]{2}-[0-9]{2}$';"
    },

    {
        "input": "Find total revenue generated from Electronics category.",
        "query": "SELECT SUM(CAST(quantity AS UNSIGNED) * CAST(price AS UNSIGNED)) FROM orders WHERE category = 'Electronics' AND quantity REGEXP '^[0-9]+$';"
    },

    {
        "input": "List orders with missing category.",
        "query": "SELECT * FROM orders WHERE category IS NULL;"
    },

    {
        "input": "Find duplicate orders based on order ID.",
        "query": "SELECT order_id, COUNT(*) FROM orders GROUP BY order_id HAVING COUNT(*) > 1;"
    },

    {
        "input": "Get all orders where quantity is not a number.",
        "query": "SELECT * FROM orders WHERE quantity NOT REGEXP '^[0-9]+$';"
    },

    {
        "input": "Find total number of orders per country after cleaning spaces.",
        "query": "SELECT TRIM(country), COUNT(*) FROM orders GROUP BY TRIM(country);"
    },

    {
        "input": "Get the highest price of any product.",
        "query": "SELECT MAX(CAST(price AS UNSIGNED)) FROM orders;"
    },

    {
        "input": "Find all valid orders where both quantity and price are numeric.",
        "query": "SELECT * FROM orders WHERE quantity REGEXP '^[0-9]+$' AND price REGEXP '^[0-9]+$';"
    }

]

In [25]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder,FewShotChatMessagePromptTemplate,PromptTemplate

example_prompt = ChatPromptTemplate.from_messages(
     [
         ("human", "{input}\nSQLQuery:"),
         ("ai", "{query}"),
     ]
 )
few_shot_prompt = FewShotChatMessagePromptTemplate(
     example_prompt=example_prompt,
     examples=examples,
     # input_variables=["input","top_k"],
     input_variables=["input"],
 )
print(few_shot_prompt.format(input1="Get the highest price of any product."))

Human: List all unique customer names after removing extra spaces.
SQLQuery:
AI: SELECT DISTINCT TRIM(customer_name) FROM orders;
Human: Find all orders where the country is India.
SQLQuery:
AI: SELECT * FROM orders WHERE TRIM(country) = 'India';
Human: Get all valid orders with proper date format YYYY-MM-DD.
SQLQuery:
AI: SELECT * FROM orders WHERE order_date REGEXP '^[0-9]{4}-[0-9]{2}-[0-9]{2}$';
Human: Find total revenue generated from Electronics category.
SQLQuery:
AI: SELECT SUM(CAST(quantity AS UNSIGNED) * CAST(price AS UNSIGNED)) FROM orders WHERE category = 'Electronics' AND quantity REGEXP '^[0-9]+$';
Human: List orders with missing category.
SQLQuery:
AI: SELECT * FROM orders WHERE category IS NULL;
Human: Find duplicate orders based on order ID.
SQLQuery:
AI: SELECT order_id, COUNT(*) FROM orders GROUP BY order_id HAVING COUNT(*) > 1;
Human: Get all orders where quantity is not a number.
SQLQuery:
AI: SELECT * FROM orders WHERE quantity NOT REGEXP '^[0-9]+$';
Human: Find to